# ***`Libraries`***

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models


# ***`Read Data`***

In [ ]:
df = pd.read_excel('Preprocessed Dataset.xlsx')
df

,Followers,Total Revenue,Years Joined,Rating Quality,Good Review,Med Review,Bad Review
0,0.060135,0.008283,1.806134,0.453583,0.631159,1.618767,1.080557
1,-0.071209,-0.047449,-1.814746,0.037897,-0.497662,1.196371,0.308262
2,-0.009052,0.008283,0.599174,0.424516,0.641790,-0.172147,1.292087
3,-0.046442,0.008283,0.599174,-0.066678,-0.096943,0.975558,1.761924
4,-0.072807,-0.053651,-0.607786,0.144730,0.344560,-0.269972,0.509950
...,...,...,...,...,...,...,...
1794,-0.072008,-0.054545,-0.004306,0.165504,-0.858078,-0.724797,-0.707270
1795,-0.075683,-0.054932,-1.814746,-2.227588,-1.622587,-0.724797,-0.707270
1796,-0.075683,-0.054932,-1.211266,-3.055399,-2.303363,-0.724797,-0.707270
1797,-0.071129,-0.052300,-1.814746,-0.478144,-0.076643,-0.724797,-0.707270


In [ ]:
X = df.values.astype(np.float32)
print("Data shape:", X.shape)

Data shape: (1799, 7)


# ***`MVDC`***

In [ ]:
# Thiết lập seed để có kết quả tái lập
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# Thiết lập các siêu tham số cho dữ liệu dạng bảng với 7 features
input_dim = X.shape[1]      # = 7
latent_dim = 4              # Không gian ẩn nhỏ hơn
num_clusters = 5            # Số cụm mong muốn (có thể điều chỉnh)
batch_size = 128
learning_rate = 1e-3
epochs = 100


# ***`3.3.1 Representation Learning (autoencoders)`***

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import numpy as np

# =============================================================================
# 1) Xây dựng Autoencoder cho View1 (Fully-connected DEKM-style)
# Mục tiêu: Học biểu diễn h_i^(1) = f₁(x_i) và tái tạo lại x_i theo Eq. (5)
# =============================================================================
def build_ae_view1(input_dim, latent_dim):
    # Tạo input layer với số chiều bằng input_dim (ví dụ: 7)
    inputs = tf.keras.Input(shape=(input_dim,))  # x_i

    # ----- Encoder f₁: Chuyển đổi x_i sang không gian latent h_i^(1) -----
    x = layers.Dense(14, activation='relu')(inputs)
    x = layers.Dense(8, activation='relu')(x)
    latent = layers.Dense(latent_dim, activation='relu')(x)
    encoder = models.Model(inputs, latent, name='encoder_fc')

    # ----- Decoder g₁: Phục hồi lại x_i từ không gian latent h_i^(1) -----
    # Tạo input layer cho decoder nhận vector latent (kích thước latent_dim)
    latent_inputs = tf.keras.Input(shape=(latent_dim,))
    x = layers.Dense(8, activation='relu')(latent_inputs)
    x = layers.Dense(14, activation='relu')(x)
    outputs = layers.Dense(input_dim, activation='relu')(x)
    decoder = models.Model(latent_inputs, outputs, name='decoder_fc')

    # ----- Autoencoder hoàn chỉnh cho View1 -----
    # Kết nối encoder và decoder: x_i -> f₁(x_i) -> g₁(f₁(x_i))
    autoencoder = models.Model(inputs, decoder(encoder(inputs)), name='autoencoder_fc')
    return encoder, decoder, autoencoder

# =============================================================================
# 2) Xây dựng Autoencoder cho View2 (U-Net style với skip connections)
# Mục tiêu: Học biểu diễn h_i^(2) = f₂(x_i) và tái tạo lại x_i qua các skip connections
# =============================================================================
def build_ae_view2(input_dim, latent_dim):
    # Tạo input layer cho x_i
    inputs = tf.keras.Input(shape=(input_dim,))  # x_i

    # ----- Encoder f₂: -----
    e1 = layers.Dense(14, activation='relu')(inputs)  # e1 có kích thước 14
    e2 = layers.Dense(8, activation='relu')(e1)         # e2 có kích thước 8
    latent = layers.Dense(latent_dim, activation='relu')(e2)
    encoder = models.Model(inputs, latent, name='encoder_unet')

    # ----- Decoder g₂: -----
    d1 = layers.Dense(8, activation='relu')(latent)
    d1_concat = layers.Concatenate()([d1, e2])
    d2 = layers.Dense(14, activation='relu')(d1_concat)
    d2_concat = layers.Concatenate()([d2, e1])
    outputs = layers.Dense(input_dim, activation='relu')(d2_concat)
    autoencoder = models.Model(inputs, outputs, name='autoencoder_unet')

    # Với View2, không tách riêng decoder vì các skip connections đã được tích hợp trong autoencoder
    return encoder, None, autoencoder

In [ ]:
# =============================================================================
# 3) Custom Training Loop: Huấn luyện đồng thời 2 autoencoder theo tổng reconstruction loss
# Mục tiêu: Tối ưu L₁ = Σᵢ (||xᵢ - g₁(f₁(xᵢ))||² + ||xᵢ - g₂(f₂(xᵢ))||²) theo Eq. (5)
# =============================================================================
def train_joint_ae(X, encoder1, decoder1, autoencoder1,
                   encoder2, autoencoder2,
                   epochs=100, batch_size=128, learning_rate=1e-3):
    # Tạo optimizer Adam
    optimizer = optimizers.Adam(learning_rate=learning_rate)
    # Chuyển dữ liệu X từ NumPy thành tf.data.Dataset, xáo trộn và chia thành các batch
    dataset = tf.data.Dataset.from_tensor_slices(X).shuffle(buffer_size=1024).batch(batch_size)

    for epoch in range(epochs):
        total_loss = 0.0  # Tích lũy loss cho epoch hiện tại
        count = 0         # Đếm số mẫu đã xử lý

        # Lặp qua từng batch
        for x_batch in dataset:
            with tf.GradientTape() as tape:
                # --- Tính toán reconstruction cho View1 ---
                # Dữ liệu x_batch đi qua encoder1 và decoder1 (View1)
                z1 = encoder1(x_batch, training=True)         # h_i^(1)
                recon1 = decoder1(z1, training=True)            # x̂₁
                # Tính loss MSE cho View1: ||x - x̂₁||²
                loss1 = tf.reduce_mean(tf.square(x_batch - recon1))

                # --- Tính toán reconstruction cho View2 ---
                # Ở View2, sử dụng toàn bộ autoencoder2 vì có skip connections tích hợp
                recon2 = autoencoder2(x_batch, training=True)   # x̂₂
                # Tính loss MSE cho View2: ||x - x̂₂||²
                loss2 = tf.reduce_mean(tf.square(x_batch - recon2))

                # Tổng loss cho batch = loss View1 + loss View2 (theo Eq. (5))
                loss = loss1 + loss2

            # Tính gradient cho các biến trainable của autoencoder1 và autoencoder2
            grads = tape.gradient(loss, autoencoder1.trainable_weights + autoencoder2.trainable_weights)
            # Cập nhật trọng số cho các model bằng optimizer
            optimizer.apply_gradients(zip(grads, autoencoder1.trainable_weights + autoencoder2.trainable_weights))

            # Tích lũy loss của batch (nhân với số mẫu của batch)
            batch_size_actual = tf.shape(x_batch)[0]
            total_loss += loss * tf.cast(batch_size_actual, tf.float32)
            count += batch_size_actual

        # Tính loss trung bình của epoch
        epoch_loss = total_loss / tf.cast(count, tf.float32)
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}: Reconstruction Loss L₁ = {epoch_loss.numpy():.4f}")

    print("Huấn luyện hoàn tất!")

In [ ]:
# =============================================================================
# 4) Ví dụ chạy toàn bộ pipeline Representation Learning
# =============================================================================

# Khởi tạo autoencoder cho View1 (fully-connected DEKM-style)
encoder_fc, decoder_fc, autoencoder_fc = build_ae_view1(input_dim, latent_dim)
# Khởi tạo autoencoder cho View2 (U-Net style với skip connections)
encoder_unet, _, autoencoder_unet = build_ae_view2(input_dim, latent_dim)

# In ra kiến trúc của hai autoencoder để kiểm tra
autoencoder_fc.summary()
autoencoder_unet.summary()

# Huấn luyện đồng thời 2 autoencoder qua custom training loop theo tổng reconstruction loss L₁ (Eq. (5))
train_joint_ae(X, encoder_fc, decoder_fc, autoencoder_fc,
               encoder_unet, autoencoder_unet,
               epochs=100, batch_size=32, learning_rate=1e-3)

print("Huấn luyện hoàn tất!")

# =============================================================================
# 5) Sau khi huấn luyện, trích xuất embedding từ 2 view và hợp nhất chúng theo Eq. (4)
# =============================================================================
# Trích xuất embedding từ View1: h_i^(1) = f₁(x_i)
h1 = encoder_fc(X)
# Trích xuất embedding từ View2: h_i^(2) = f₂(x_i)
h2 = encoder_unet(X)
# Hợp nhất hai embedding theo công thức: h_i = 1/2 * (h_i^(1) + h_i^(2))
h_fused = 0.5 * (h1 + h2)
print("Fused embedding shape:", h_fused.shape)

Model: "autoencoder_fc"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 7)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_fc (Functional)         │ (None, 4)              │           268 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_fc (Functional)         │ (None, 7)              │           271 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 539 (2.11 KB)

 Trainable params: 539 (2.11 KB)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 14)        │        112 │ input_layer_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 8)         │        120 │ dense_18[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 4)         │         36 │ dense_19[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 8)         │         40 │ dense_20[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 16)        │          0 │ dense_21[0][0],   │
│ (Concatenate)       │                   │            │ dense_19[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_22 (Dense)    │ (None, 14)        │        238 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 28)        │          0 │ dense_22[0][0],   │
│ (Concatenate)       │                   │            │ dense_18[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_23 (Dense)    │ (None, 7)         │        203 │ concatenate_3[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 749 (2.93 KB)

 Trainable params: 749 (2.93 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 10/100: Reconstruction Loss L₁ = 0.8325
Epoch 20/100: Reconstruction Loss L₁ = 0.8298
Epoch 30/100: Reconstruction Loss L₁ = 0.8287
Epoch 40/100: Reconstruction Loss L₁ = 0.8279
Epoch 50/100: Reconstruction Loss L₁ = 0.8269
Epoch 60/100: Reconstruction Loss L₁ = 0.8262
Epoch 70/100: Reconstruction Loss L₁ = 0.8258
Epoch 80/100: Reconstruction Loss L₁ = 0.8257
Epoch 90/100: Reconstruction Loss L₁ = 0.8256
Epoch 100/100: Reconstruction Loss L₁ = 0.8255
Huấn luyện hoàn tất!
Huấn luyện hoàn tất!
Fused embedding shape: (1799, 4)


# ***`3.3.2. clustering in the fused embedding space (K-means objective)`***

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

def cluster_with_init(h_fused, n_clusters, init_method):
    """
    Hàm thực hiện phân cụm các vector embedding h_fused với thuật toán KMeans
    sử dụng phương pháp khởi tạo init_method.

    Tham số:
      - h_fused: Tensor hoặc mảng numpy có kích thước (n_samples, latent_dim).
      - n_clusters: Số cụm mong muốn.
      - init_method: Phương pháp khởi tạo, ví dụ "random" hoặc "k-means++".

    Trả về:
      - cluster_assignments: Các nhãn cụm cho từng sample.
      - centroids: Tâm của các cụm.
      - L2: Tổng khoảng cách bình phương giữa các điểm và tâm cụm.
      - sil_score: Chỉ số Silhouette (dựa trên khoảng cách Euclidean).
    """
    # Chuyển h_fused về NumPy nếu cần
    h_np = h_fused.numpy() if isinstance(h_fused, tf.Tensor) else h_fused

    # Thiết lập KMeans với số cụm n_clusters, phương pháp khởi tạo init_method, n_init=10
    kmeans = KMeans(n_clusters=n_clusters, init=init_method, random_state=42)

    # Phân cụm và lấy nhãn cho mỗi sample
    cluster_assignments = kmeans.fit_predict(h_np)
    centroids = kmeans.cluster_centers_

    # Tính tổng loss L₂: tổng khoảng cách bình phương giữa các điểm với tâm cụm của chúng.
    L2 = 0.0
    for j in range(n_clusters):
        indices = np.where(cluster_assignments == j)[0]
        if len(indices) > 0:
            diff = h_np[indices] - centroids[j]
            L2 += np.sum(np.linalg.norm(diff, axis=1)**2)

    # Tính chỉ số Silhouette score
    sil_score = silhouette_score(h_np, cluster_assignments, metric='euclidean')

    return cluster_assignments, centroids, L2, sil_score

def grid_search_init(h_fused, n_clusters, init_methods=["random", "k-means++"]):

    results = {}
    best_score = -1
    best_init = None

    for method in init_methods:
        _, centroids, L2, sil_score = cluster_with_init(h_fused, n_clusters, method)
        results[method] = {"silhouette_score": sil_score, "L2_loss": L2}
        print(f"Init method: {method}, Silhouette score: {sil_score:.4f}, Total L2 loss: {L2:.4f}")

        if sil_score > best_score:
            best_score = sil_score
            best_init = method

    print("\nPhương pháp khởi tạo tối ưu:", best_init)
    print("Silhouette score:", best_score)
    return best_init, best_score, results

n_clusters = 5
best_init, best_score, grid_results = grid_search_init(h_fused, n_clusters,
                                                        init_methods=["random", "k-means++"])


Init method: random, Silhouette score: 0.3432, Total L2 loss: 152.4172
Init method: k-means++, Silhouette score: 0.3951, Total L2 loss: 153.0159

Phương pháp khởi tạo tối ưu: k-means++
Silhouette score: 0.39506996


# ***`3.3.3. Orthonormal Transformation using Eigenvectors`***

In [ ]:
import numpy as np

def orthonormal_transformation(h_fused, cluster_assignments, centroids):
    """
    Thực hiện bước orthonormal transformation (theo Eq. (7)-(9) trong bài báo).

    Tham số:
    - h_fused: embedding hợp nhất (numpy array), shape = (n_samples, latent_dim).
    - cluster_assignments: mảng chỉ số cụm (length = n_samples).
    - centroids: tâm cụm, shape = (n_clusters, latent_dim).

    Trả về:
    - y: embedding đã được biến đổi (shape = (n_samples, latent_dim)).
    - V: ma trận trực giao lấy từ các eigenvectors của S_w, shape = (latent_dim, latent_dim).
    - L3: giá trị L_3 = Tr(V S_w V^T).
    """

    # Convert h_fused to NumPy array if it's a TensorFlow tensor
    h_fused = h_fused.numpy() if isinstance(h_fused, tf.Tensor) else h_fused
    n_samples, latent_dim = h_fused.shape
    n_clusters = centroids.shape[0]

    # 1) Tính ma trận scatter trong cụm: S_w = sum_{j=1..k} sum_{h_i in C_j} (h_i - mu_j)(h_i - mu_j)^T
    S_w = np.zeros((latent_dim, latent_dim), dtype=np.float64)
    for j in range(n_clusters):
        # Lấy index các điểm thuộc cụm j
        indices = np.where(cluster_assignments == j)[0]
        if len(indices) > 0:
            # Tâm cụm j
            mu_j = centroids[j]
            # Tính (h_i - mu_j) cho tất cả h_i thuộc cụm j
            diff = h_fused[indices] - mu_j  # shape = (num_points_in_cluster, latent_dim)
            # Cộng dồn diff^T * diff cho S_w
            # diff.T: shape (latent_dim, num_points_in_cluster)
            # => diff.T @ diff: shape (latent_dim, latent_dim)
            S_w += diff.T @ diff

    # 2) Tính eigen-decomposition của S_w
    #    eigenvalues, eigenvectors = np.linalg.eig(S_w) hoặc np.linalg.eigh(S_w)
    #    eigh() đảm bảo giá trị riêng là thực (S_w đối xứng)
    eigenvalues, eigenvectors = np.linalg.eigh(S_w)

    # 3) Sắp xếp eigenvectors theo thứ tự tăng dần của eigenvalues
    #    (theo bài báo: eigenvector cuối cùng "đóng góp ít nhất")
    idx = np.argsort(eigenvalues)  # sắp xếp giá trị riêng theo thứ tự tăng
    eigenvalues_sorted = eigenvalues[idx]
    V = eigenvectors[:, idx]       # ma trận trực giao (từng cột là 1 eigenvector)

    # 4) Biến đổi embedding: y_i = h_i * V (trong code: y = h_fused @ V)
    y = h_fused @ V

    # 5) Tính loss L3 = Tr(V S_w V^T)
    #    (theo Eq. (9) trong bài báo)
    L3 = np.trace(V.T @ S_w @ V)

    return y, V, L3

# ==============================
# Ví dụ sử dụng:
# Giả sử ta đã có:
# - h_fused: embedding hợp nhất (n_samples, latent_dim)
# - cluster_assignments: mảng gán cụm
# - centroids: tâm cụm (n_clusters, latent_dim)
#   => Từ bước 3.3.2 (K-means) ta có cluster_assignments & centroids
# ==============================
# Call orthonormal_transformation to define y
y, V, L3 = orthonormal_transformation(h_fused, cluster_assignments, centroids) # Defining y here

print("Ma trận trực giao V shape:", V.shape)
print("Embedding sau biến đổi (y) shape:", y.shape) # Using y here
print("Giá trị L3 = Tr(V S_w V^T):", L3)

Ma trận trực giao V shape: (4, 4)
Embedding sau biến đổi (y) shape: (1799, 4)
Giá trị L3 = Tr(V S_w V^T): 2263.2548213005075


# ***`3.3.4. greedy optimization for enhanced clustering`***

In [ ]:
import numpy as np

def greedy_optimization(y, cluster_assignments, centroids):
    """
    Thực hiện bước greedy optimization để điều chỉnh chiều cuối cùng của mỗi điểm y_i,
    đưa nó về gần tâm cụm ở chiều cuối đó.
    Tính toán L4 = sum_{j=1 to k} sum_{y_i in C_j} ||y_i - y'_i||^2,
    trong đó y'_i là phiên bản đã chỉnh sửa.

    Tham số:
    - y: ma trận embedding đã qua bước orthonormal transformation, shape (n_samples, latent_dim).
    - cluster_assignments: mảng chỉ số cụm (length = n_samples).
    - centroids: mảng tâm cụm (shape = (n_clusters, latent_dim)).

    Trả về:
    - y_adjusted: ma trận y sau khi đã điều chỉnh chiều cuối.
    - L4: tổng loss L4 = sum ||y_i - y'_i||^2.
    """
    # Sao chép y để tránh sửa trực tiếp
    y_adjusted = np.copy(y)
    L4 = 0.0

    n_clusters = centroids.shape[0]

    # Lặp qua từng cụm
    for j in range(n_clusters):
        # Lấy index các điểm thuộc cụm j
        indices = np.where(cluster_assignments == j)[0]
        if len(indices) == 0:
            continue

        # Tâm cụm j
        centroid_j = centroids[j]
        # Lấy giá trị tâm cụm ở chiều cuối
        centroid_last_dim = centroid_j[-1]

        # Điều chỉnh chiều cuối cho từng điểm trong cụm j
        for idx in indices:
            original_val = y_adjusted[idx, -1]         # y_i[-1] trước khi sửa
            # Gán chiều cuối của y_i thành chiều cuối của tâm cụm
            y_adjusted[idx, -1] = centroid_last_dim

            # Tính ||y_i - y'_i||^2 (khoảng cách trong toàn bộ không gian)
            diff_vector = y[idx] - y_adjusted[idx]
            L4 += np.sum(diff_vector**2)

    return y_adjusted, L4


# ================================
# Ví dụ sử dụng:
# Giả sử:
#   y: embedding sau bước orthonormal_transform (shape = (n_samples, latent_dim))
#   cluster_assignments: mảng nhãn cụm
#   centroids: tâm cụm (shape = (n_clusters, latent_dim))
# ================================

y_adjusted, L4 = greedy_optimization(y, cluster_assignments, centroids)

print("Sau bước greedy optimization, L4 =", L4)


Sau bước greedy optimization, L4 = 1226.4070226550018


# ***`3.3.5. joint optimization`***

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import optimizers

# Giả sử ta có sẵn các hàm đã định nghĩa:
# - get_autoencoders_loss(X, model_view1, model_view2) -> L1
# - cluster_fused_embeddings(h_fused, n_clusters) -> (cluster_assignments, centroids, L2)
# - orthonormal_transformation(h_fused, cluster_assignments, centroids) -> (y, V, L3)
# - greedy_optimization(y, cluster_assignments, centroids) -> (y_adjusted, L4)

# --- Hàm tính reconstruction loss (L₁) của hai autoencoder ---
def get_autoencoders_loss(x, autoencoder1, autoencoder2):
    """
    Tính tổng reconstruction loss của hai autoencoder (View1 và View2):
      L1 = MSE(x, autoencoder1(x)) + MSE(x, autoencoder2(x))
    """
    recon1 = autoencoder1(x, training=True)  # x̂₁ từ autoencoder View1
    recon2 = autoencoder2(x, training=True)  # x̂₂ từ autoencoder View2
    loss1 = tf.reduce_mean(tf.square(x - recon1))
    loss2 = tf.reduce_mean(tf.square(x - recon2))
    return loss1 + loss2


# --- Hàm train_joint_optimization: Joint optimization theo L = L1 + L2 + L3 + L4 ---
def train_joint_optimization(
    X,              # Dữ liệu đầu vào, shape (n_samples, input_dim)
    model_view1,    # Autoencoder View1 (model_view1 = autoencoder_fc)
    model_view2,    # Autoencoder View2 (model_view2 = autoencoder_unet)
    encoder1,       # Encoder của View1 (encoder_fc)
    encoder2,       # Encoder của View2 (encoder_unet)
    n_clusters=5,
    max_iter=50,
    batch_size=256,
    learning_rate=1e-3
):
    # Tạo optimizer cho hai autoencoder
    optimizer = optimizers.Adam(learning_rate=learning_rate)

    # Chuyển dữ liệu X thành tf.data.Dataset với shuffle và batch
    dataset = tf.data.Dataset.from_tensor_slices(X).shuffle(buffer_size=1024).batch(batch_size)

    # Biến để lưu L1 trung bình của mỗi epoch
    epoch_L1_loss = 0.0
    total_samples = X.shape[0]

    for epoch in range(max_iter):
        total_loss_epoch = 0.0
        batch_count = 0

        # --- Bước 1: Tối ưu L₁ qua các batch ---
        for x_batch in dataset:
            with tf.GradientTape() as tape:
                # Tính reconstruction loss L1 cho batch hiện tại
                L1 = get_autoencoders_loss(x_batch, model_view1, model_view2)
            # Tính gradient qua các weights của hai autoencoder
            grads = tape.gradient(L1, model_view1.trainable_weights + model_view2.trainable_weights)
            optimizer.apply_gradients(zip(grads, model_view1.trainable_weights + model_view2.trainable_weights))

            # Tích lũy loss cho epoch
            batch_size_actual = tf.shape(x_batch)[0]
            total_loss_epoch += L1 * tf.cast(batch_size_actual, tf.float32)
            batch_count += batch_size_actual

        # Tính L1 trung bình trên toàn tập X trong epoch này
        epoch_L1_loss = total_loss_epoch / tf.cast(batch_count, tf.float32)

        # --- Bước 2: Tính clustering objective L₂ ---
        # Trích xuất embedding từ hai encoder và hợp nhất theo Eq. (4):
        h1 = encoder1(X)  # h_i^(1)
        h2 = encoder2(X)  # h_i^(2)
        h_fused = 0.5 * (h1 + h2)  # h_i = 1/2 (h_i^(1) + h_i^(2))

        # Lấy kết quả phân cụm, L2 loss từ bước clustering (với hàm cluster_fused_embeddings đã định nghĩa)
        cluster_assignments, centroids, L2, _ = cluster_with_init(h_fused, n_clusters, init_method="k-means++")

        # --- Bước 3: Orthonormal transformation: Tính L₃ ---
        y, V, L3 = orthonormal_transformation(h_fused, cluster_assignments, centroids)

        # --- Bước 4: Greedy optimization: Tính L₄ ---
        y_adjusted, L4 = greedy_optimization(y, cluster_assignments, centroids)

        # --- Gộp toàn bộ loss ---
        # L1 là tensor, các L2, L3, L4 là số float (numpy), chuyển chúng về tf.constant
        additional_loss = tf.constant(L2 + L3 + L4, dtype=tf.float32)
        total_loss = epoch_L1_loss + additional_loss

        # In thông tin của epoch
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{max_iter}: L1 = {epoch_L1_loss.numpy():.4f}, "
                  f"L2 = {L2:.4f}, L3 = {L3:.4f}, L4 = {L4:.4f}, Total Loss = {total_loss.numpy():.4f}")

    print("Joint optimization hoàn tất!")
    return

# =========================
# GIẢ SỬ TA ĐÃ CÓ:
# - X: (n_samples, input_dim)
# - model_view1, model_view2: 2 autoencoder (hoặc 2 MLP)
# - encoder1, encoder2: 2 encoder tách riêng
# - get_autoencoders_loss(): hàm tính L1 = L1_view1 + L1_view2
# - cluster_fused_embeddings(): hàm tính L2
# - orthonormal_transformation(): hàm tính L3
# - greedy_optimization(): hàm tính L4
# =========================

# Gọi hàm train_joint_optimization(X, model_view1, model_view2, encoder1, encoder2, ...)
train_joint_optimization(
    X,              # Dữ liệu đầu vào
    model_view1=autoencoder_fc,
    model_view2=autoencoder_unet,
    encoder1=encoder_fc,
    encoder2=encoder_unet,
    n_clusters=5,
    max_iter=50,
    batch_size=32,
    learning_rate=1e-3
)

Epoch 5/50: L1 = 0.8255, L2 = 151.6341, L3 = 151.6341, L4 = 1017.1061, Total Loss = 1321.1998
Epoch 10/50: L1 = 0.8254, L2 = 152.0196, L3 = 152.0196, L4 = 1079.5927, Total Loss = 1384.4574
Epoch 15/50: L1 = 0.8254, L2 = 152.6697, L3 = 152.6697, L4 = 1099.8507, Total Loss = 1406.0155
Epoch 20/50: L1 = 0.8254, L2 = 155.4346, L3 = 155.4346, L4 = 978.6444, Total Loss = 1290.3390
Epoch 25/50: L1 = 0.8254, L2 = 157.4747, L3 = 157.4747, L4 = 3975.1039, Total Loss = 4290.8784
Epoch 30/50: L1 = 0.8253, L2 = 160.4028, L3 = 160.4028, L4 = 1361.1287, Total Loss = 1682.7596
Epoch 35/50: L1 = 0.8254, L2 = 165.3493, L3 = 165.3493, L4 = 4343.8489, Total Loss = 4675.3726
Epoch 40/50: L1 = 0.8253, L2 = 164.3683, L3 = 164.3683, L4 = 765.8992, Total Loss = 1095.4612
Epoch 45/50: L1 = 0.8253, L2 = 166.3967, L3 = 166.3967, L4 = 3482.3896, Total Loss = 3816.0083
Epoch 50/50: L1 = 0.8253, L2 = 168.8258, L3 = 168.8258, L4 = 1287.2851, Total Loss = 1625.7621
Joint optimization hoàn tất!


In [ ]:
from sklearn.metrics import silhouette_score
import numpy as np
import tensorflow as tf

# Giả sử các biến sau đã tồn tại sau khi hoàn thiện quá trình joint optimization:
# - encoder_fc (encoder của View1)
# - encoder_unet (encoder của View2)
# - X: dữ liệu đầu vào, shape = (n_samples, input_dim)
# - n_clusters: Số cụm mong muốn
# - best_init: phương pháp khởi tạo tối ưu đã được tìm qua grid search (ví dụ: "k-means++")

# Sau khi joint optimization, ta trích xuất embedding từ 2 view:
h1_final = encoder_fc(X)       # h_i^(1): embedding từ View1 (shape: (n_samples, latent_dim))
h2_final = encoder_unet(X)     # h_i^(2): embedding từ View2 (shape: (n_samples, latent_dim))

# Hợp nhất embedding theo Eq. (4): h_i = 1/2 * (h_i^(1) + h_i^(2))
h_fused_final = 0.5 * (h1_final + h2_final)

# Phân cụm các embedding hợp nhất bằng KMeans với phương pháp khởi tạo best_init
cluster_assignments, centroids, L2_loss, sil_score_final = cluster_with_init(h_fused_final, n_clusters, init_method=best_init)

# In ra các chỉ số đánh giá phân cụm:
print("Performance Evaluation:")
print("Final Silhouette score: {:.4f}".format(sil_score_final))
print("Final Total L2 loss: {:.4f}".format(L2_loss))


Performance Evaluation:
Final Silhouette score: 0.4078
Final Total L2 loss: 168.8258


# ***`Thêm biến nhị phân từ FP-Max (Forward Selection)`***

In [ ]:
# Giả sử:
# - h_fused: embedding hợp nhất có kích thước (n_samples, latent_dim)
# - binary_features: mảng các biến nhị phân, shape = (n_samples, n_binary)

selected_features = h_fused  # Bắt đầu với fused embedding

# Forward selection qua các biến nhị phân
n_binary = binary_features.shape[1]
current_silhouette = silhouette_score(selected_features.numpy() if isinstance(selected_features, tf.Tensor) else selected_features,
                                        cluster_with_init(selected_features, n_clusters, init_method="k-means++")[0])
print("Silhouette ban đầu:", current_silhouette)

for i in range(n_binary):
    # Thêm biến nhị phân thứ i vào selected_features
    candidate_feature = binary_features[:, i:i+1]  # shape (n_samples, 1)
    new_features = np.concatenate([selected_features, candidate_feature], axis=1)

    # Phân cụm trên tập mở rộng này (sử dụng phương pháp khởi tạo đã chọn hoặc mặc định)
    candidate_assignments, _, _, candidate_silhouette = cluster_with_init(new_features, n_clusters, init_method="k-means++")

    # Nếu Silhouette score cải thiện, giữ lại biến; nếu không thì không thêm biến đó
    if candidate_silhouette > current_silhouette:
        selected_features = new_features
        current_silhouette = candidate_silhouette
        print(f"Biến nhị phân thứ {i} được giữ lại; Silhouette tăng thành: {current_silhouette:.4f}")
    else:
        print(f"Biến nhị phân thứ {i} không được giữ lại; Silhouette: {candidate_silhouette:.4f}")

# Sau forward selection, sử dụng selected_features cho bước clustering cuối cùng:
final_cluster_assignments, final_centroids, final_L2, final_silhouette = cluster_with_init(selected_features, n_clusters, init_method="k-means++")
print("Kết quả phân cụm cuối cùng với các biến nhị phân được chọn:")
print("Silhouette score:", final_silhouette)


In [ ]:
# Giả sử bạn đã có h_fused và binary_features từ các bước trước
# h_fused: numpy array or tf.Tensor có kích thước (n_samples, latent_dim)
# binary_features: numpy array có kích thước (n_samples, n_binary)

# Nếu h_fused là tf.Tensor thì chuyển về numpy
if isinstance(h_fused, tf.Tensor):
    h_fused_np = h_fused.numpy()
else:
    h_fused_np = h_fused

# Tạo X_extended: nối h_fused (continuous) với binary_features (categorical)
X_extended = np.concatenate([h_fused_np, binary_features], axis=1)

# Xác định chỉ số của các cột categorical trong X_extended:
# Giả sử h_fused có latent_dim cột (là continuous), các cột tiếp theo là biến nhị phân.
n_cont = h_fused_np.shape[1]
n_binary = binary_features.shape[1]
categorical_columns = list(range(n_cont, n_cont + n_binary))
print("Categorical columns indices:", categorical_columns)

# Sử dụng K-Prototypes từ thư viện kmodes:
from kmodes.kprototypes import KPrototypes

# Số cụm mong muốn (ví dụ: num_clusters, bạn có thể thay đổi)
num_clusters = 5

# Khởi tạo KPrototypes; sử dụng init='Huang' (thường được xem tương đương cho việc dùng k-means++ cho dữ liệu categorical)
# Tham số gamma điều chỉnh trọng số giữa các biến continuous và categorical
kproto = KPrototypes(n_clusters=num_clusters, init='Cao', random_state=42, gamma=1)

# Chạy phân cụm với K-Prototypes; truyền vào mảng dữ liệu X_extended và danh sách chỉ số categorical
cluster_labels = kproto.fit_predict(X_extended, categorical=categorical_columns)
print("Cluster labels from K-Prototypes:", cluster_labels)

# Lấy các tâm cụm từ model
centroids = kproto.cluster_centroids_  # Đây là danh sách [centroids_cont, centroids_cat]
print("Centroids (continuous part) shape:", np.array(centroids[0]).shape)
print("Centroids (categorical part) shape:", np.array(centroids[1]).shape)

# Tính ma trận khoảng cách Gower cho dữ liệu hỗn hợp
# Cài đặt gower: pip install gower
import gower
D = gower.gower_matrix(X_extended)

# Tính Silhouette score trên ma trận khoảng cách đã tính (với metric 'precomputed')
from sklearn.metrics import silhouette_score
sil_score = silhouette_score(D, cluster_labels, metric="precomputed")
print("Silhouette score (Gower distance):", sil_score)
